# Imports

In [322]:
import pandas as pd
import numpy as np
import igraph as ig
from collections import defaultdict
import os



# Part 1

In [147]:
df = pd.read_csv("data/Part_A/1/balanced_graph.csv")
g = ig.Graph.TupleList(df[["u","v"]].itertuples(index=False), directed=False)
g.es["sign"] = df["sign"].tolist()
triangles = g.cliques(min=3, max=3)

In [148]:
zero_edges = [e.index for e in g.es if e["sign"] == 0]
triangle_edges = []
for tri in triangles:
    n1, n2, n3 = tri
    e1 = g.get_eid(n1, n2)
    e2 = g.get_eid(n2, n3)
    e3 = g.get_eid(n3, n1)
    triangle_edges.append((e1, e2, e3))



In [149]:
def is_triangle_balanced(tri_edges, graph):
    prod = 1
    for e in tri_edges:
        s = graph.es[e]["sign"]
        if s == 0:
            return True  # unknown edges can still be assigned
        prod *= s
    return prod > 0


In [150]:
def backtrack_balance(graph, zero_edges, triangle_edges, idx=0):
    if idx == len(zero_edges):
        for tri in triangle_edges:
            if not is_triangle_balanced(tri, graph):
                return False
        return True

    edge_idx = zero_edges[idx]

    for sign in [1, -1]:        
        graph.es[edge_idx]["sign"] = sign        
        valid = True
        for tri in triangle_edges:
            if edge_idx in tri and not is_triangle_balanced(tri, graph):
                valid = False
                break
        if valid:
            if backtrack_balance(graph, zero_edges, triangle_edges, idx + 1):
                return True        
        graph.es[edge_idx]["sign"] = 0
    return False


In [151]:
success = backtrack_balance(g, zero_edges, triangle_edges)
if success:
    print("Graph successfully balanced!")
else:
    print("No assignment can fully balance the graph with given constraints.")


Graph successfully balanced!


# Part B

In [323]:

def draw_clusters(graph, membership, graph_name,path):
    num_clusters = len(set(membership))
    palette = ig.drawing.colors.ClusterColoringPalette(num_clusters)
    graph.vs["color"] = [palette[m] for m in membership]
        
    graph.es["color"] = ["#3498db" if s > 0 else "#e74c3c" for s in graph.es["sign"]]
    graph.es["width"] = [2 if s > 0 else 1 for s in graph.es["sign"]]
    

    graph.vs["label"] = graph.vs["name"]
    graph.vs["label_size"] = 10
    
    layout = graph.layout("fr") 
    
    filename = os.path.join(path,f"clusters_{graph_name}.png")
    
    ig.plot(graph, filename, layout=layout, bbox=(600, 600), margin=50)
    print(f"Visualization saved as: {filename}")

def analyze_structural_balance(graph, graph_name, weak=False,path=""):
    print(f"\n--- Analyzing: {graph_name} ---")
    
    g_pos = graph.copy()
    neg_edge_indices = [e.index for e in graph.es if e['sign'] == -1]
    g_pos.delete_edges(neg_edge_indices)
    
    clusters = g_pos.components()
    membership = clusters.membership
    num_supernodes = len(clusters)
    print(f"Identified {num_supernodes} super-nodes (factions).")
        
    neg_edges = [e for e in graph.es if e['sign'] == -1]
    for edge in neg_edges:
        u, v = edge.tuple
        if membership[u] == membership[v]:
            u_name = graph.vs[u]['name']
            v_name = graph.vs[v]['name']
            print("Result: UNBALANCED")
            print(f"Reason: Internal contradiction. '{u_name}' and '{v_name}' are friends but have a negative edge.")
            print_supernode_assignments(graph, membership)
            draw_clusters(graph, membership, graph_name,path)
            return False

    reduced_graph = ig.Graph(num_supernodes)
    reduced_edges = []
    for edge in neg_edges:
        u, v = edge.tuple
        if membership[u] != membership[v]:
            reduced_edges.append((membership[u], membership[v]))
            
    reduced_graph.add_edges(reduced_edges)
    reduced_graph.simplify()
    
    is_bipartite = reduced_graph.is_bipartite()
    
    if is_bipartite:
        print("Result: Strongly BALANCED")
        print("Reason: Reduced graph is bipartite.")
    elif weak:
        print("Result: Weakly BALANCED")
        print("Reason: No inner-cluster negative edges (but reduced graph is not bipartite).")
    else:
        print("Result: UNBALANCED")
        print("Reason: Reduced graph contains an odd cycle.")

    print_supernode_assignments(graph, membership)
    draw_clusters(graph, membership, graph_name,path)
    return is_bipartite or weak

def print_supernode_assignments(graph, membership):
    print("\nSuper-node Assignments:")
    groups = defaultdict(list)
    for node_idx, cluster_id in enumerate(membership):        
        node_label = graph.vs[node_idx]['name'] 
        groups[cluster_id].append(node_label)
    
    for cluster_id, nodes in sorted(groups.items()):
        print(f"  Super-node {cluster_id} (Size {len(nodes)}): {nodes}")

In [ ]:
for i in ["a","b","c","d","e","f","g","h"]:
    df = pd.read_csv(f"data/Part_A/2/network_{i}.csv")
    graph = ig.Graph.TupleList(df[["u","v"]].itertuples(index=False), directed=False)
    graph.es["sign"] = df["sign"].tolist()
    analyze_structural_balance(graph,f"network_{i}",weak=False,path="./q4_data/q4_partb_images/")




--- Analyzing: network_a ---
Identified 3 super-nodes (factions).
Result: Strongly BALANCED
Reason: Reduced graph is bipartite.

Super-node Assignments:
  Super-node 0 (Size 28): [0, 2, 4, 6, 9, 27, 29, 1, 22, 26, 15, 31, 32, 33, 34, 3, 7, 8, 19, 24, 12, 20, 5, 16, 25, 11, 30, 13]
  Super-node 1 (Size 4): [10, 14, 21, 23]
  Super-node 2 (Size 3): [17, 18, 28]
Visualization saved as: ./q4_partb_images/clusters_network_a.png

--- Analyzing: network_b ---
Identified 3 super-nodes (factions).
Result: UNBALANCED
Reason: Internal contradiction. '0' and '14' are friends but have a negative edge.

Super-node Assignments:
  Super-node 0 (Size 36): [0, 3, 7, 9, 11, 14, 21, 28, 30, 34, 35, 36, 1, 13, 19, 32, 2, 5, 24, 20, 26, 27, 29, 31, 4, 37, 18, 10, 25, 33, 8, 12, 17, 23, 16, 22]
  Super-node 1 (Size 1): [6]
  Super-node 2 (Size 1): [15]
Visualization saved as: ./q4_partb_images/clusters_network_b.png

--- Analyzing: network_c ---
Identified 3 super-nodes (factions).
Result: UNBALANCED
Reason

# Part C

In [327]:
for i in ["a","b","c","d","e"]:
    df = pd.read_csv(f"data/Part_A/3/network_{i}.csv")
    graph = ig.Graph.TupleList(df[["u","v"]].itertuples(index=False), directed=False)
    graph.es["sign"] = df["sign"].tolist()
    analyze_structural_balance(graph,f"network_{i}",weak=True,path="./q4_data/q4_partc_images/")




--- Analyzing: network_a ---
Identified 3 super-nodes (factions).
Result: Strongly BALANCED
Reason: Reduced graph is bipartite.

Super-node Assignments:
  Super-node 0 (Size 5): [4, 7, 12, 2, 0]
  Super-node 1 (Size 5): [1, 3, 13, 11, 10]
  Super-node 2 (Size 4): [9, 5, 8, 6]
Visualization saved as: ./q4_data/q4_partc_images/clusters_network_a.png

--- Analyzing: network_b ---
Identified 4 super-nodes (factions).
Result: Weakly BALANCED
Reason: No inner-cluster negative edges (but reduced graph is not bipartite).

Super-node Assignments:
  Super-node 0 (Size 5): [14, 10, 9, 0, 8]
  Super-node 1 (Size 5): [5, 15, 12, 18, 2]
  Super-node 2 (Size 5): [7, 6, 13, 17, 3]
  Super-node 3 (Size 5): [19, 1, 11, 16, 4]
Visualization saved as: ./q4_data/q4_partc_images/clusters_network_b.png

--- Analyzing: network_c ---
Identified 4 super-nodes (factions).
Result: Weakly BALANCED
Reason: No inner-cluster negative edges (but reduced graph is not bipartite).

Super-node Assignments:
  Super-node 0